# Preparação dos dados para detecção de anomalias

Prepara o Fake.br para os notebooks de detecção de anomalias (02_IF, 03_OCSVM, 04_LOF).
Os detectores aprendem o padrão de notícias **True (0)** e sinalizam textos com comportamento
linguístico atípico. Notícias **Fake (1)** ficam fora do ajuste e da calibração.

> **Interpretação:** `anomalyScore` é um score de anomalia, não uma probabilidade de falsidade.
> Um alerta indica necessidade de revisão e não comprova que a notícia seja falsa.

Ao final, este notebook salva `dados_preparados.pkl` (partições e features) e `pipelines.pkl`
(pipelines ainda não ajustados), que os notebooks dos modelos carregam.

## Imports

In [1]:
import re
import unicodedata
from itertools import combinations
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

RANDOM_STATE = 42

## Carregamento dos dados

O ZIP do Fake.br, em uma revisão fixa, é baixado para `data/` quando ainda não existe.
Textos completos e metadados são carregados; o truncamento acontece antes da extração
das features. Nenhuma notícia é excluída por comprimento.

In [2]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
dataFolder = Path("data")
dataFolder.mkdir(exist_ok=True)
archivePath = dataFolder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archivePath.exists():
    temporaryPath = archivePath.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporaryPath.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    with ZipFile(temporaryPath) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporaryPath.replace(archivePath)

In [3]:
metadataColumns = [
    "autor", "link", "categoria", "data_publicacao",
    "num_tokens", "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos", "num_adjetivos",
    "num_adverbios", "num_verbos_modais", "num_pron_1_2_sing", "num_pron_1_plural",
    "num_pronomes", "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade", "diversidade",
]
numericMetadataColumns = metadataColumns[4:]


def loadNewsTexts(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue
        articleId = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": articleId, "text": archive.read(name).decode("utf-8"),
            "label": label, "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def loadNewsMetadata(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}-meta-information/" not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadataColumns), f"Esquema inesperado em {name}: {len(values)} linhas"
        articleId = Path(name).name.removesuffix("-meta.txt") + ("t" if label == 0 else "")
        records.append({**dict(zip(metadataColumns, values)), "id": articleId, "metadataLabel": label})
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)

In [4]:
with ZipFile(archivePath) as archive:
    textsFrame = pd.concat([
        loadNewsTexts(archive, "fake", 1), loadNewsTexts(archive, "true", 0),
    ], ignore_index=True)
    metadataFrame = pd.concat([
        loadNewsMetadata(archive, "fake", 1), loadNewsMetadata(archive, "true", 0),
    ], ignore_index=True)

assert textsFrame["id"].is_unique and metadataFrame["id"].is_unique
newsFrame = textsFrame.merge(metadataFrame, on="id", validate="one_to_one")
assert len(newsFrame) == len(textsFrame) == len(metadataFrame), "Texto/metadados sem correspondência"
assert newsFrame["label"].eq(newsFrame["metadataLabel"]).all()
newsFrame = newsFrame.drop(columns="metadataLabel")
assert newsFrame["text"].str.strip().ne("").all()
print(f"{len(newsFrame):,} notícias carregadas; textos e metadados correspondem 1:1.")

7,200 notícias carregadas; textos e metadados correspondem 1:1.


## Contrato dos labels

**0 = True; 1 = Fake.** O rótulo forma as partições e permite avaliação externa.
Não entra como feature. A origem nas pastas também é verificada.

In [5]:
assert set(newsFrame["label"].unique()) == {0, 1}
assert newsFrame["sourceClass"].eq(newsFrame["label"].map({0: "true", 1: "fake"})).all()
newsFrame.groupby(["label", "sourceClass"]).size().rename("newsCount").to_frame()

,,newsCount
label,sourceClass,
0,true,3600
1,fake,3600


## Tratamento dos metadados

Valores não numéricos viram NaN com contagem explícita; nenhum ausente é preenchido antes
do treino. `tem_autor` indica ausência de autor para string vazia, `None`, `none` ou `NULL`.
Não é uma medida de credibilidade.

In [6]:
convertedMetadata = newsFrame[numericMetadataColumns].apply(pd.to_numeric, errors="coerce")
display(convertedMetadata.isna().sum().rename("missingAfterNumericConversion").to_frame())
newsFrame[numericMetadataColumns] = convertedMetadata
newsFrame["tem_autor"] = (~newsFrame["autor"].fillna("").astype(str).str.strip().isin(
    ["", "None", "none", "NULL"]
)).astype(int)

,missingAfterNumericConversion
num_tokens,0
num_palavras,0
num_types,0
num_links,1393
num_maiusculas,0
num_verbos,0
num_verbos_subj_imp,0
num_substantivos,0
num_adjetivos,0
num_adverbios,0


## Extração das features no texto truncado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros `CHARACTER_LIMIT` caracteres,
incluindo espaços/pontuação. Textos menores permanecem menores, sem preenchimento. O corte
pode dividir palavras/frases.

- Palavras: sequências de letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
- Tipos: palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade = tipos/palavras.
- Maiúsculas: palavras totalmente maiúsculas com mais de uma letra.
- Links: URLs http(s) presentes no corpo.

As demais contagens linguísticas dos metadados (calculadas no texto inteiro) são marcadas como
ausentes na cópia de features. Autor permanece como metadado válido da notícia inteira.

In [7]:
CHARACTER_LIMIT = 300
anomalyColumns = ["tem_autor", "typeTokenRatio", "linkDensity", "punctuationDensity", "uppercaseRatio", "diversidade"]
wordPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
tokenPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)


def calculateRatio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safeDenominator = denominator.where(denominator.gt(0) & np.isfinite(denominator))
    return numerator.div(safeDenominator).replace([np.inf, -np.inf], np.nan)


def measureText(text, characterLimit):
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")[:characterLimit]
    words = wordPattern.findall(normalized)
    return {"text": normalized, "num_palavras": len(words),
            "num_tokens": len(tokenPattern.findall(normalized)),
            "num_types": len({word.casefold() for word in words}),
            "num_links": len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE)),
            "num_maiusculas": sum(word.isupper() and len(word) > 1 for word in words),
            "num_caracteres": len(normalized)}


def extractAnomalyFeatures(newsFrame, characterLimit=CHARACTER_LIMIT):
    featuresFrame = newsFrame.copy(deep=True)
    featuresFrame["num_palavras_original"] = newsFrame["num_palavras"]
    featuresFrame["text_original"] = newsFrame["text"]
    featuresFrame[numericMetadataColumns] = np.nan
    measurements = pd.DataFrame([measureText(text, characterLimit) for text in newsFrame["text"]], index=newsFrame.index)
    featuresFrame[measurements.columns] = measurements
    featuresFrame["typeTokenRatio"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_tokens"])
    featuresFrame["diversidade"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_palavras"])
    featuresFrame["linkDensity"] = calculateRatio(featuresFrame["num_links"], featuresFrame["num_palavras"])
    featuresFrame["punctuationDensity"] = calculateRatio(featuresFrame["num_tokens"] - featuresFrame["num_palavras"], featuresFrame["num_tokens"])
    featuresFrame["uppercaseRatio"] = calculateRatio(featuresFrame["num_maiusculas"], featuresFrame["num_palavras"])
    return featuresFrame

In [8]:
featuresFrame = extractAnomalyFeatures(newsFrame)
display(featuresFrame[anomalyColumns].isna().sum().rename("missingCount").to_frame())
display(featuresFrame.groupby("label")[["num_palavras_original", "num_palavras", "num_caracteres"]].agg(["min", "median", "max"]))
print("Textos menores que o limite:", int(featuresFrame["num_caracteres"].lt(CHARACTER_LIMIT).sum()))

,missingCount
tem_autor,0
typeTokenRatio,0
linkDensity,0
punctuationDensity,0
uppercaseRatio,0
diversidade,0


num_palavras_original              num_palavras             \
                        min median   max          min median max   
label                                                              
0                        20  922.0  7588           19   47.0  60   
1                         9  157.0  2294            8   48.0  64   

      num_caracteres              
                 min median  max  
label                             
0                114  300.0  300  
1                 46  300.0  300

Textos menores que o limite: 99


## Sanity checks

As seis features usam o prefixo truncado; a presença de autor vem dos metadados.
`num_palavras` fica na extração para auditoria e não alimenta o modelo.

In [9]:
assert not {"num_palavras", "label", "id"} & set(anomalyColumns)
assert len(set(anomalyColumns)) == 6
assert featuresFrame["id"].is_unique
assert featuresFrame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert featuresFrame["num_palavras_original"].equals(newsFrame["num_palavras"])
assert featuresFrame["num_verbos"].isna().all()
assert not np.isinf(featuresFrame[anomalyColumns].to_numpy(dtype=float)).any()
assert measureText("casa " * 100, 300)["num_palavras"] == 60
assert measureText("casa " * 100, 300)["num_caracteres"] == 300

## Análise estatística

Estatísticas descritivas por label. Esta inspeção é exploratória: não é usada para selecionar
features, hiperparâmetros ou threshold. Correlações de Pearson com o comprimento aparecem no
total e por classe para não confundir efeitos de classe e de tamanho. NaN em correlação pode
indicar feature constante.

In [10]:
labelNames = {0: "True", 1: "Fake"}
display(featuresFrame.groupby("label")[anomalyColumns].describe().T)

lengthColumns = ["num_palavras", "num_tokens"]
lengthCorrelations = pd.concat({
    name: group[anomalyColumns + lengthColumns].corr().loc[anomalyColumns, lengthColumns]
    for name, group in [("All", featuresFrame)] + [
        (f"{labelNames[label]} ({label})", group) for label, group in featuresFrame.groupby("label")
    ]
}, names=["group", "feature"])
display(lengthCorrelations)
display(featuresFrame[anomalyColumns].corr().round(3))

label                               0            1
tem_autor          count  3600.000000  3600.000000
                   mean      0.979722     0.020000
                   std       0.140968     0.140019
                   min       0.000000     0.000000
                   25%       1.000000     0.000000
                   50%       1.000000     0.000000
                   75%       1.000000     0.000000
                   max       1.000000     1.000000
typeTokenRatio     count  3600.000000  3600.000000
                   mean      0.698048     0.664573
                   std       0.068560     0.066960
                   min       0.358974     0.413333
                   25%       0.655172     0.621212
                   50%       0.701754     0.666667
                   75%       0.745455     0.707095
                   max       0.921569     0.900000
linkDensity        count  3600.000000  3600.000000
                   mean      0.000000     0.000000
                   std       0.000000     0.000000
                   min       0.000000     0.000000
                   25%       0.000000     0.000000
                   50%       0.000000     0.000000
                   75%       0.000000     0.000000
                   max       0.000000     0.000000
punctuationDensity count  3600.000000  3600.000000
                   mean      0.145714     0.167808
                   std       0.048265     0.056673
                   min       0.000000     0.035714
                   25%       0.111111     0.127273
                   50%       0.142857     0.166667
                   75%       0.175439     0.203390
                   max       0.384615     0.470588
uppercaseRatio     count  3600.000000  3600.000000
                   mean      0.019995     0.022792
                   std       0.023448     0.032375
                   min       0.000000     0.000000
                   25%       0.000000     0.000000
                   50%       0.020000     0.019608
                   75%       0.025000     0.039216
                   max       0.315789     0.420000
diversidade        count  3600.000000  3600.000000
                   mean      0.817017     0.799228
                   std       0.065109     0.067613
                   min       0.551020     0.545455
                   25%       0.777778     0.755102
                   50%       0.822222     0.800000
                   75%       0.862745     0.844444
                   max       1.000000     1.000000

num_palavras  num_tokens
group    feature                                     
All      tem_autor              -0.016428   -0.158099
         typeTokenRatio         -0.107702   -0.485101
         linkDensity                  NaN         NaN
         punctuationDensity     -0.234614    0.436514
         uppercaseRatio         -0.029622    0.019670
         diversidade            -0.323344   -0.273850
True (0) tem_autor               0.023761    0.025026
         typeTokenRatio         -0.068439   -0.500045
         linkDensity                  NaN         NaN
         punctuationDensity     -0.241752    0.528841
         uppercaseRatio         -0.064469    0.010045
         diversidade            -0.255571   -0.245689
Fake (1) tem_autor               0.019447   -0.078203
         typeTokenRatio         -0.133990   -0.463896
         linkDensity                  NaN         NaN
         punctuationDensity     -0.249768    0.362342
         uppercaseRatio         -0.015290    0.013255
         diversidade            -0.375238   -0.274395

,tem_autor,typeTokenRatio,linkDensity,punctuationDensity,uppercaseRatio,diversidade
tem_autor,1.000,0.245,NaN,-0.221,-0.057,0.128
typeTokenRatio,0.245,1.000,NaN,-0.594,-0.088,0.780
linkDensity,NaN,NaN,NaN,NaN,NaN,NaN
punctuationDensity,-0.221,-0.594,NaN,1.000,0.073,0.036
uppercaseRatio,-0.057,-0.088,NaN,0.073,1.000,-0.051
diversidade,0.128,0.780,NaN,0.036,-0.051,1.000


## Split de treino, validação e teste

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `random_state=42`. **Nenhuma notícia Fake participa do ajuste ou da calibração.**

Limitação: a divisão é por notícia, não por assunto, fonte ou data. O corpus possui pares
True/Fake com o mesmo número-base; IDs `123t` e `123` são notícias distintas, mas podem tratar
do mesmo assunto em partições diferentes.

In [11]:
normalFrame = featuresFrame.loc[featuresFrame["label"].eq(0)]
fakeFrame = featuresFrame.loc[featuresFrame["label"].eq(1)]
normalTrainFrame, normalHoldoutFrame = train_test_split(normalFrame, train_size=0.60, random_state=RANDOM_STATE)
normalValidationFrame, normalTestFrame = train_test_split(normalHoldoutFrame, test_size=0.50, random_state=RANDOM_STATE)
fakeValidationFrame, fakeTestFrame = train_test_split(fakeFrame, test_size=0.50, random_state=RANDOM_STATE)
partitions = {
    "normalTrain": normalTrainFrame, "normalValidation": normalValidationFrame,
    "normalTest": normalTestFrame, "fakeValidation": fakeValidationFrame,
    "fakeTest": fakeTestFrame,
}

for name, frame in partitions.items():
    assert not frame.empty
    assert frame["label"].eq(0 if name.startswith("normal") else 1).all()
for (leftName, left), (rightName, right) in combinations(partitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"]), f"IDs compartilhados: {leftName}/{rightName}"
assert sum(len(frame) for frame in partitions.values()) == len(featuresFrame)
pd.Series({name: len(frame) for name, frame in partitions.items()}, name="count").to_frame()

,count
normalTrain,2160
normalValidation,720
normalTest,720
fakeValidation,1800
fakeTest,1800


## Pipelines

Todos os pipelines imputam pela mediana do treino. Isolation Forest não usa `StandardScaler`;
One-Class SVM e LOF, por dependerem de distâncias, usam. OCSVM e LOF têm uma matriz fechada
de candidatos, escolhidos na validação pelos notebooks dos modelos.

- `IF` - Isolation Forest com 300 árvores
- `OCSVM` - One-Class SVM RBF com `nu` ∈ {0.01, 0.025, 0.05, 0.10}
- `LOF` - Local Outlier Factor (`novelty=True`) com `n_neighbors` ∈ {10, 20, 40, 80}

In [12]:
def buildPipeline(detector, scale):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps + [("detector", detector)])


pipelines = {
    "IF": buildPipeline(
        IsolationForest(n_estimators=300, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1),
        scale=False,
    ),
    "OCSVM": {
        key: buildPipeline(OneClassSVM(kernel="rbf", gamma="scale", nu=nu), scale=True)
        for key, nu in {"ocsvm_nu_001": 0.01, "ocsvm_nu_0025": 0.025, "ocsvm_nu_005": 0.05, "ocsvm_nu_010": 0.10}.items()
    },
    "LOF": {
        f"lof_neighbors_{neighbors:03d}": buildPipeline(
            LocalOutlierFactor(n_neighbors=neighbors, contamination="auto", novelty=True, n_jobs=1),
            scale=True,
        )
        for neighbors in [10, 20, 40, 80]
    },
}
{name: list(value) if isinstance(value, dict) else value for name, value in pipelines.items()}

{'IF': Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                 ('detector',
                  IsolationForest(n_estimators=300, n_jobs=-1,
                                  random_state=42))]),
 'OCSVM': ['ocsvm_nu_001', 'ocsvm_nu_0025', 'ocsvm_nu_005', 'ocsvm_nu_010'],
 'LOF': ['lof_neighbors_010',
  'lof_neighbors_020',
  'lof_neighbors_040',
  'lof_neighbors_080']}

In [13]:
joblib.dump((partitions, anomalyColumns), "./dados_preparados.pkl")
joblib.dump(pipelines, "./pipelines.pkl")

['./pipelines.pkl']